# Processing Time Analysis by Prompt Strategy

This notebook analyzes the processing time performance across different prompt strategies:
- **base_version**: Basic version with minimal context
- **with_geom**: Version with geospatial features  
- **with_geom_time**: Version with geospatial features and temporal analysis (most advanced)

The analysis focuses on the `processing_time` column from prediction CSV files.

**Current Status**: Analysis supports all three core strategies: base_version, with_geom, and with_geom_time for comprehensive performance comparison.

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import os
import gc
import psutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Dict, List, Tuple, Optional
import pickle
import hashlib
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ============= SIMPLIFIED CONFIGURATION FOR CANVA EXPORT =============
# MODEL SELECTION - Change this to analyze specific model
MODEL = "mistral_7b"

# Strategy configuration - Core strategies only
EXPECTED_STRATEGIES = {
    'base_version': 'Base',
    'with_geom': 'Geom',
    'with_geom_time': 'Time'
}

# Anchor configuration
ANCHOR_DIRS = ['middle', 'penultimate']

# Performance configuration
MAX_WORKERS = 4  # Parallel file processing
BATCH_SIZE = 1000  # Records per batch before DataFrame creation
CHUNK_SIZE = 5000  # CSV reading chunk size

# Cache configuration
USE_CACHE = True
CACHE_DIR = Path('cache')

# Memory management settings
pd.set_option('mode.chained_assignment', None)

# Set paths
base_path = Path('/leonardo_work/IscrC_LLM-Mob/LLM-Mob-As-Mobility-Interpreter')
results_path = base_path / 'results'
cache_path = base_path / 'notebook' / CACHE_DIR

# Ensure cache directory exists
cache_path.mkdir(parents=True, exist_ok=True)

def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def get_cache_key(model: str) -> str:
    """Generate cache key for model data"""
    cache_inputs = [model, str(EXPECTED_STRATEGIES), str(ANCHOR_DIRS)]
    return hashlib.md5(''.join(cache_inputs).encode()).hexdigest()

print(f"=== PROCESSING TIME ANALYSIS FOR CANVA EXPORT - MODEL: {MODEL} ===")
print(f"Configuration:")
print(f"  - Core strategies: {len(EXPECTED_STRATEGIES)} ({', '.join(EXPECTED_STRATEGIES.keys())})")
print(f"  - Output: canva_with_units_{MODEL}.csv")
print(f"Base path: {base_path}")
print(f"Results path: {results_path}")
print(f"✅ Simplified analysis focused on Canva export generation")
print(f"Initial memory usage: {get_memory_usage():.1f} MB")

=== PROCESSING TIME ANALYSIS FOR CANVA EXPORT - MODEL: mistral_7b ===
Configuration:
  - Core strategies: 3 (base_version, with_geom, with_geom_time)
  - Output: canva_with_units_mistral_7b.csv
Base path: /leonardo_work/IscrC_LLM-Mob/LLM-Mob-As-Mobility-Interpreter
Results path: /leonardo_work/IscrC_LLM-Mob/LLM-Mob-As-Mobility-Interpreter/results
✅ Simplified analysis focused on Canva export generation
Initial memory usage: 805.8 MB


In [18]:
# ============= ENHANCED DATA LOADING WITH OPTIMIZATIONS =============

class ProcessingTimeAnalyzer:
    """Enhanced processing time analyzer with caching, validation and parallel processing"""
    
    def __init__(self, model: str, base_path: Path):
        self.model = model
        self.base_path = base_path
        self.results_path = base_path / 'results'
        self.cache_path = cache_path
        self.file_stats = defaultdict(int)
        self.record_stats = defaultdict(int)
        
    def validate_path_structure(self) -> Tuple[bool, List[str]]:
        """Validate that the expected directory structure exists"""
        validation_errors = []
        
        if not self.results_path.exists():
            validation_errors.append(f"Results directory not found: {self.results_path}")
            return False, validation_errors
            
        for anchor_type in ANCHOR_DIRS:
            model_path = self.results_path / anchor_type / self.model
            
            if not model_path.exists():
                validation_errors.append(f"Model directory not found: {model_path}")
                continue
                
            # Check for strategy directories
            existing_strategies = {d.name for d in model_path.iterdir() if d.is_dir()}
            
            # Core strategies validation
            missing_core = set(EXPECTED_STRATEGIES.keys()) - existing_strategies
            if missing_core:
                validation_errors.append(f"Missing core strategies in {model_path}: {missing_core}")
            
            # Check for CSV files in each existing strategy
            for strategy in existing_strategies:
                if strategy in EXPECTED_STRATEGIES:  # Only check core strategies
                    strategy_path = model_path / strategy
                    csv_files = list(strategy_path.glob('*.csv'))
                    csv_files = [f for f in csv_files if not f.name.endswith('_checkpoint.txt')]
                    
                    if not csv_files:
                        validation_errors.append(f"No CSV files found in {strategy_path}")
                        
        return len(validation_errors) == 0, validation_errors
    
    def get_cache_filepath(self) -> Path:
        """Get cache file path for current model"""
        cache_key = get_cache_key(self.model)
        return self.cache_path / f"processing_times_{self.model}_{cache_key}.pkl"
        
    def save_to_cache(self, data: pd.DataFrame) -> None:
        """Save processed data to cache"""
        if not USE_CACHE:
            return
            
        cache_file = self.get_cache_filepath()
        try:
            with open(cache_file, 'wb') as f:
                pickle.dump({
                    'data': data,
                    'model': self.model,
                    'timestamp': pd.Timestamp.now(),
                    'file_stats': dict(self.file_stats),
                    'record_stats': dict(self.record_stats)
                }, f)
            print(f"✅ Data cached to: {cache_file}")
        except Exception as e:
            print(f"⚠️ Failed to save cache: {e}")
    
    def load_from_cache(self) -> Optional[pd.DataFrame]:
        """Load processed data from cache if available"""
        if not USE_CACHE:
            return None
            
        cache_file = self.get_cache_filepath()
        if not cache_file.exists():
            return None
            
        try:
            with open(cache_file, 'rb') as f:
                cached = pickle.load(f)
                
            print(f"✅ Loaded from cache: {cache_file}")
            print(f"   Cache timestamp: {cached['timestamp']}")
            print(f"   Cached records: {len(cached['data']):,}")
            
            # Restore stats
            self.file_stats = defaultdict(int, cached.get('file_stats', {}))
            self.record_stats = defaultdict(int, cached.get('record_stats', {}))
            
            return cached['data']
            
        except Exception as e:
            print(f"⚠️ Failed to load cache: {e}")
            return None
    
    def process_single_file(self, file_info: Dict) -> Tuple[List[Dict], int, str]:
        """Process a single CSV file with enhanced error handling"""
        csv_file = Path(file_info['path'])
        model_name = file_info['model']
        strategy_label = file_info['strategy']
        anchor_type = file_info['anchor']
        
        records = []
        total_records = 0
        error_msg = ""
        
        try:
            # Validate file exists and is readable
            if not csv_file.exists():
                return [], 0, f"File not found: {csv_file}"
                
            if csv_file.stat().st_size == 0:
                return [], 0, f"Empty file: {csv_file}"
            
            # Read file in chunks with specific error handling
            try:
                chunk_iter = pd.read_csv(
                    csv_file, 
                    chunksize=CHUNK_SIZE,
                    on_bad_lines='skip', 
                    engine='python'
                )
            except pd.errors.EmptyDataError:
                return [], 0, f"Empty data in file: {csv_file.name}"
            except pd.errors.ParserError as e:
                return [], 0, f"Parser error in {csv_file.name}: {str(e)[:100]}"
            except UnicodeDecodeError as e:
                return [], 0, f"Encoding error in {csv_file.name}: {str(e)[:100]}"
            
            # Process chunks
            for chunk_idx, chunk in enumerate(chunk_iter):
                try:
                    # Validate required columns
                    if 'processing_time' not in chunk.columns:
                        if chunk_idx == 0:  # Only warn once per file
                            error_msg = f"Missing 'processing_time' column in {csv_file.name}"
                        continue
                    
                    # Filter valid processing times
                    valid_chunk = chunk.dropna(subset=['processing_time'])
                    
                    # Validate data types
                    try:
                        valid_chunk['processing_time'] = pd.to_numeric(valid_chunk['processing_time'], errors='coerce')
                        valid_chunk = valid_chunk.dropna(subset=['processing_time'])
                    except Exception as e:
                        if chunk_idx == 0:
                            error_msg = f"Invalid processing_time format in {csv_file.name}: {str(e)[:100]}"
                        continue
                    
                    if len(valid_chunk) == 0:
                        continue
                    
                    # Filter reasonable processing times (0.1s to 1000s)
                    valid_chunk = valid_chunk[
                        (valid_chunk['processing_time'] >= 0.1) & 
                        (valid_chunk['processing_time'] <= 1000)
                    ]
                    
                    if len(valid_chunk) == 0:
                        continue
                    
                    # Extract dataset name
                    dataset_name = csv_file.stem.split('_pred_')[0] if '_pred_' in csv_file.stem else csv_file.stem
                    
                    # Batch process records for better performance
                    batch_records = [
                        {
                            'strategy': strategy_label,
                            'model': model_name,
                            'anchor': anchor_type,
                            'dataset': dataset_name,
                            'file': csv_file.name,
                            'processing_time': float(row['processing_time'])
                        }
                        for _, row in valid_chunk.iterrows()
                    ]
                    
                    records.extend(batch_records)
                    total_records += len(batch_records)
                    
                except Exception as e:
                    error_msg = f"Chunk processing error in {csv_file.name}: {str(e)[:100]}"
                    continue
        
        except PermissionError:
            error_msg = f"Permission denied: {csv_file}"
        except MemoryError:
            error_msg = f"Memory error processing: {csv_file.name}"
        except Exception as e:
            error_msg = f"Unexpected error in {csv_file.name}: {str(e)[:100]}"
        
        return records, total_records, error_msg
    
    def collect_file_list(self) -> List[Dict]:
        """Collect list of all CSV files to process"""
        file_list = []
        
        for anchor_type in ANCHOR_DIRS:
            model_path = self.results_path / anchor_type / self.model
            
            if not model_path.exists():
                print(f"⚠️ Skipping missing directory: {model_path}")
                continue
            
            strategy_dirs = [d for d in model_path.iterdir() if d.is_dir()]
            
            for strategy_dir in strategy_dirs:
                strategy = strategy_dir.name
                
                # Only process core strategies
                if strategy in EXPECTED_STRATEGIES:
                    strategy_label = EXPECTED_STRATEGIES[strategy]
                    
                    csv_files = list(strategy_dir.glob('*.csv'))
                    csv_files = [f for f in csv_files if not f.name.endswith('_checkpoint.txt')]
                    
                    print(f"📁 Found {len(csv_files)} CSV files in {anchor_type}/{self.model}/{strategy}")
                    
                    for csv_file in csv_files:
                        file_list.append({
                            'path': str(csv_file),
                            'model': self.model,
                            'strategy': strategy_label,
                            'anchor': anchor_type,
                            'filename': csv_file.name
                        })
                else:
                    print(f"⚠️ Ignoring strategy: {strategy} in {anchor_type}")
        
        return file_list
    
    def load_all_processing_times_optimized(self) -> pd.DataFrame:
        """Load all processing times with optimizations"""
        
        # Try loading from cache first
        cached_data = self.load_from_cache()
        if cached_data is not None:
            print("🚀 Using cached data")
            return cached_data
        
        # Validate directory structure
        is_valid, errors = self.validate_path_structure()
        if not is_valid:
            print("❌ Directory structure validation failed:")
            for error in errors:
                print(f"   - {error}")
            print("⚠️ Continuing with available data...")
        
        # Collect all files to process
        file_list = self.collect_file_list()
        
        if not file_list:
            print("❌ No files found to process")
            return pd.DataFrame()
        
        print(f"📁 Found {len(file_list)} files to process")
        print(f"🔧 Processing with {MAX_WORKERS} parallel workers...")
        
        # Process files in parallel with progress bar
        all_dataframes = []
        successful_files = 0
        failed_files = 0
        
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            # Submit all file processing tasks
            future_to_file = {
                executor.submit(self.process_single_file, file_info): file_info 
                for file_info in file_list
            }
            
            # Process completed tasks with progress bar
            with tqdm(total=len(file_list), desc="Processing files", unit="file") as pbar:
                batch_records = []
                
                for future in as_completed(future_to_file):
                    file_info = future_to_file[future]
                    
                    try:
                        records, record_count, error_msg = future.result()
                        
                        if error_msg:
                            print(f"⚠️ {error_msg}")
                            failed_files += 1
                        else:
                            successful_files += 1
                            
                        if records:
                            batch_records.extend(records)
                            
                            # Update statistics
                            strategy = file_info['strategy']
                            anchor = file_info['anchor']
                            self.file_stats[f"{strategy}_{anchor}"] += 1
                            self.record_stats[strategy] += record_count
                            
                            # Create DataFrame in batches to manage memory
                            if len(batch_records) >= BATCH_SIZE:
                                df_batch = pd.DataFrame(batch_records)
                                all_dataframes.append(df_batch)
                                batch_records.clear()
                                gc.collect()  # Free memory
                                
                        pbar.set_postfix({
                            'Success': successful_files, 
                            'Failed': failed_files,
                            'Memory': f"{get_memory_usage():.0f}MB"
                        })
                        
                    except Exception as e:
                        print(f"❌ Task execution error for {file_info['filename']}: {e}")
                        failed_files += 1
                    
                    pbar.update(1)
                
                # Process remaining records
                if batch_records:
                    df_batch = pd.DataFrame(batch_records)
                    all_dataframes.append(df_batch)
        
        # Combine all DataFrames
        if not all_dataframes:
            print("❌ No valid data found")
            return pd.DataFrame()
        
        print(f"🔄 Combining {len(all_dataframes)} DataFrame batches...")
        
        try:
            final_df = pd.concat(all_dataframes, ignore_index=True)
            
            # Final data validation and cleanup
            final_df = final_df.dropna(subset=['processing_time'])
            final_df = final_df[final_df['processing_time'] > 0]
            
            print(f"✅ Data loading completed:")
            print(f"   - Successful files: {successful_files}")
            print(f"   - Failed files: {failed_files}")
            print(f"   - Total records: {len(final_df):,}")
            print(f"   - Memory usage: {get_memory_usage():.1f} MB")
            
            # Save to cache for future use
            self.save_to_cache(final_df)
            
            return final_df
            
        except Exception as e:
            print(f"❌ Error combining DataFrames: {e}")
            return pd.DataFrame()

# Initialize analyzer and load data
print(f"🚀 Initializing enhanced analyzer for model: {MODEL}")
analyzer = ProcessingTimeAnalyzer(MODEL, base_path)

try:
    df_times = analyzer.load_all_processing_times_optimized()
    
    if len(df_times) == 0:
        print(f"❌ No data loaded for model {MODEL}")
    else:
        print(f"\n🎉 SUCCESS! Loaded {len(df_times):,} processing time records")
        print(f"📊 Data overview:")
        print(f"   - Strategies: {sorted(df_times['strategy'].unique())}")
        print(f"   - Anchors: {sorted(df_times['anchor'].unique())}")
        print(f"   - Datasets: {len(df_times['dataset'].unique())}")
        print(f"   - Processing time range: {df_times['processing_time'].min():.3f}s - {df_times['processing_time'].max():.3f}s")
        
        # Verify strategy-anchor combinations
        combinations = df_times.groupby(['strategy', 'anchor']).size()
        print(f"\n📈 Strategy-Anchor combinations:")
        for (strategy, anchor), count in combinations.items():
            print(f"   - {strategy} × {anchor}: {count:,} records")
        
        # Memory cleanup
        gc.collect()
        print(f"💾 Final memory usage: {get_memory_usage():.1f} MB")
        
except Exception as e:
    print(f"❌ CRITICAL ERROR: {e}")
    df_times = pd.DataFrame()

🚀 Initializing enhanced analyzer for model: mistral_7b
❌ Directory structure validation failed:
   - No CSV files found in /leonardo_work/IscrC_LLM-Mob/LLM-Mob-As-Mobility-Interpreter/results/penultimate/mistral_7b/with_geom_time
⚠️ Continuing with available data...
📁 Found 12 CSV files in middle/mistral_7b/base_version
📁 Found 12 CSV files in middle/mistral_7b/with_geom
📁 Found 12 CSV files in middle/mistral_7b/with_geom_time
📁 Found 12 CSV files in penultimate/mistral_7b/base_version
📁 Found 12 CSV files in penultimate/mistral_7b/with_geom
📁 Found 0 CSV files in penultimate/mistral_7b/with_geom_time
📁 Found 60 files to process
🔧 Processing with 4 parallel workers...


Processing files: 100%|██████████| 60/60 [02:36<00:00,  2.60s/file, Success=60, Failed=0, Memory=1332MB]


🔄 Combining 60 DataFrame batches...
✅ Data loading completed:
   - Successful files: 60
   - Failed files: 0
   - Total records: 3,120,703
   - Memory usage: 1473.7 MB
✅ Data cached to: /leonardo_work/IscrC_LLM-Mob/LLM-Mob-As-Mobility-Interpreter/notebook/cache/processing_times_mistral_7b_95067c0cc9e73a92738c3e6b6246cdf5.pkl

🎉 SUCCESS! Loaded 3,120,703 processing time records
📊 Data overview:
   - Strategies: ['Base', 'Geom', 'Time']
   - Anchors: ['middle', 'penultimate']
   - Datasets: 12
   - Processing time range: 0.505s - 12.317s

📈 Strategy-Anchor combinations:
   - Base × middle: 624,505 records
   - Base × penultimate: 624,514 records
   - Geom × middle: 624,505 records
   - Geom × penultimate: 622,652 records
   - Time × middle: 624,527 records
💾 Final memory usage: 1177.7 MB


In [19]:
# Generate Canva export file - Simplified version
if len(df_times) == 0:
    print(f"ERROR: No data available for model {MODEL}. Please check the data loading step.")
else:
    print(f"=== PROCESSING TIME ANALYSIS FOR MODEL {MODEL} ===")
    print(f"Total records: {len(df_times):,}")
    
    # Basic statistics by strategy
    print("\n=== RECORDS PER STRATEGY ===")
    strategy_counts = df_times['strategy'].value_counts()
    for strategy, count in strategy_counts.items():
        print(f"{strategy}: {count:,} records")

    print("\n=== BASIC STATISTICS BY STRATEGY ===")
    strategy_stats = df_times.groupby('strategy')['processing_time'].agg([
        'count', 'mean', 'median', 'std', 'min', 'max'
    ]).round(3)
    print(strategy_stats)
    
    # ============= EXPORT FOR CANVA =============
    print("\n=== GENERATING CANVA EXPORT FILE ===")
    
    # Prepare data for Canva - Strategy-level statistics
    canva_strategy_data = []
    
    for strategy in sorted(df_times['strategy'].unique()):
        strategy_data = df_times[df_times['strategy'] == strategy]['processing_time']
        
        canva_strategy_data.append({
            'Strategy': strategy,
            'Min_Processing_Time_Seconds': f"{strategy_data.min():.3f}s",
            'Max_Processing_Time_Seconds': f"{strategy_data.max():.3f}s", 
            'Mean_Processing_Time_Seconds': f"{strategy_data.mean():.3f}s",
            'Count_Records': len(strategy_data),
        })
    
    # Create DataFrame and export the only file we need
    canva_df = pd.DataFrame(canva_strategy_data)
    
    # Export the canva_with_units file (the only one needed)
    units_output_path = base_path / 'notebook' / f'canva_with_units_{MODEL}.csv'
    canva_df.to_csv(units_output_path, index=False)
    
    print(f"✅ Canva file exported: {units_output_path}")
    print("\nGenerated data:")
    print(canva_df.to_string(index=False))
    
    print(f"\n✅ ANALYSIS COMPLETE - Canva export ready!")
    print(f"File location: {units_output_path}")
    
    # Memory cleanup
    del canva_strategy_data, canva_df
    gc.collect()
    print(f"Memory usage: {get_memory_usage():.1f} MB")

=== PROCESSING TIME ANALYSIS FOR MODEL mistral_7b ===
Total records: 3,120,703

=== RECORDS PER STRATEGY ===
Base: 1,249,019 records
Geom: 1,247,157 records
Time: 624,527 records

=== BASIC STATISTICS BY STRATEGY ===
            count   mean  median    std    min     max
strategy                                              
Base      1249019  1.573   1.426  0.458  0.532  12.317
Geom      1247157  1.285   1.222  0.305  0.505  11.672
Time       624527  1.518   1.490  0.247  0.713  11.135

=== GENERATING CANVA EXPORT FILE ===
✅ Canva file exported: /leonardo_work/IscrC_LLM-Mob/LLM-Mob-As-Mobility-Interpreter/notebook/canva_with_units_mistral_7b.csv

Generated data:
Strategy Min_Processing_Time_Seconds Max_Processing_Time_Seconds Mean_Processing_Time_Seconds  Count_Records
    Base                      0.532s                     12.317s                       1.573s        1249019
    Geom                      0.505s                     11.672s                       1.285s        1247157
 

In [20]:
# Performance Summary Report
if len(df_times) == 0:
    print("No data available for performance summary.")
else:
    print("=== PROCESSING TIME PERFORMANCE SUMMARY ===")
    print("\nStrategy Performance Ranking (by mean processing time):")

    strategy_ranking = df_times.groupby('strategy')['processing_time'].agg([
        'mean', 'median', 'count'
    ]).sort_values('mean')

    for i, (strategy, stats) in enumerate(strategy_ranking.iterrows(), 1):
        print(f"{i}. {strategy}:")
        print(f"   Mean: {stats['mean']:.2f}s")
        print(f"   Median: {stats['median']:.2f}s")
        print(f"   Records: {stats['count']:,}")
        print()

    # Efficiency metrics
    print("\n=== EFFICIENCY ANALYSIS ===")
    base_mean = strategy_ranking.loc['Base Version', 'mean'] if 'Base Version' in strategy_ranking.index else None

    if base_mean:
        print(f"Base Version mean processing time: {base_mean:.2f}s")
        print("\nOverhead compared to Base Version:")
        
        for strategy, stats in strategy_ranking.iterrows():
            if strategy != 'Base Version':
                overhead = ((stats['mean'] - base_mean) / base_mean) * 100
                print(f"{strategy}: +{overhead:.1f}% ({stats['mean'] - base_mean:.2f}s additional)")

    # Processing rate (predictions per second)
    print("\n=== PROCESSING RATE ===")
    for strategy, stats in strategy_ranking.iterrows():
        rate = 1 / stats['mean']
        print(f"{strategy}: {rate:.3f} predictions/second")

=== PROCESSING TIME PERFORMANCE SUMMARY ===

Strategy Performance Ranking (by mean processing time):
1. Geom:
   Mean: 1.28s
   Median: 1.22s
   Records: 1,247,157.0

2. Time:
   Mean: 1.52s
   Median: 1.49s
   Records: 624,527.0

3. Base:
   Mean: 1.57s
   Median: 1.43s
   Records: 1,249,019.0


=== EFFICIENCY ANALYSIS ===

=== PROCESSING RATE ===
Geom: 0.778 predictions/second
Time: 0.659 predictions/second
Base: 0.636 predictions/second
